In [ ]:
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from sklearn.metrics import confusion_matrix, classification_report

# ==================== 1. DOWNLOAD DATASET ====================
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"keo065","key":"KGAT_a6a87740ae869cf972dd3e5abe061e12"}')
os.system('chmod 600 /root/.kaggle/kaggle.json')

!kaggle datasets download -d salader/dogsvscats -p ./data --unzip

# ==================== 2. LOAD DATA ====================
train_datagen = ImageDataGenerator(rescale=1.0/255)
test_datagen = ImageDataGenerator(rescale=1.0/255)

train_data = train_datagen.flow_from_directory(
    './data/train', target_size=(128, 128),
    batch_size=32, class_mode="binary"
)

test_data = test_datagen.flow_from_directory(
    './data/test', target_size=(128, 128),
    batch_size=32, class_mode="binary", shuffle=False
)

# ==================== 3. BUILD MODEL ====================
base_model = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights="imagenet")
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
output = Dense(1, activation="sigmoid")(x)
model = Model(inputs=base_model.input, outputs=output)

# ==================== 4. PHASE 1 ====================
print("📍 PHASE 1: Feature Extraction...\n")
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history1 = model.fit(train_data, epochs=5, validation_data=test_data)

# ==================== 5. PHASE 2 ====================
print("\n🔓 PHASE 2: Fine-tuning...\n")
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy", metrics=["accuracy"]
)
history2 = model.fit(train_data, epochs=5, validation_data=test_data)

# ==================== 6. SAVE MODEL ====================
os.makedirs('/content/drive/MyDrive/dogs_vs_cats_results', exist_ok=True)
model.save('/content/drive/MyDrive/dogs_vs_cats_results/dogs_vs_cats_mobilenetv2.keras')
print("✅ Model saved to Google Drive!")

# ==================== 7. GENERATE & SAVE PLOTS ====================
# Training history plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MobileNetV2 - Dogs vs Cats', fontsize=16, fontweight='bold')

all_acc = history1.history['accuracy'] + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss = history1.history['loss'] + history2.history['loss']
all_val_loss = history1.history['val_loss'] + history2.history['val_loss']

axes[0].plot(all_acc, label='Train', linewidth=2.5, marker='o')
axes[0].plot(all_val_acc, label='Validation', linewidth=2.5, marker='s')
axes[0].axvline(x=5, color='red', linestyle='--', alpha=0.5, label='Fine-tuning starts')
axes[0].set_title('Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(all_loss, label='Train', linewidth=2.5, marker='o')
axes[1].plot(all_val_loss, label='Validation', linewidth=2.5, marker='s')
axes[1].axvline(x=5, color='red', linestyle='--', alpha=0.5, label='Fine-tuning starts')
axes[1].set_title('Loss', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dogs_vs_cats_results/training_history.png', dpi=300)
print("✅ Training history saved!")

# Confusion matrix plot
test_data.reset()
predictions = model.predict(test_data, verbose=0)
predicted_classes = (predictions > 0.5).astype(int).reshape(-1)
true_classes = test_data.classes

cm = confusion_matrix(true_classes, predicted_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Cat', 'Dog'],
            yticklabels=['Cat', 'Dog'])
plt.title(f'Confusion Matrix - Dogs vs Cats', fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dogs_vs_cats_results/confusion_matrix.png', dpi=300)
print("✅ Confusion matrix saved!")

# ==================== 8. PRINT FINAL RESULTS ====================
test_data.reset()
test_loss, test_acc = model.evaluate(test_data, verbose=0)
print(f"\n{'='*60}")
print(f"🎉 FINAL RESULTS")
print(f"{'='*60}")
print(f"✅ Test Accuracy: {test_acc*100:.2f}%")
print(f"✅ Test Loss: {test_loss:.4f}")
print(f"\n{classification_report(true_classes, predicted_classes, target_names=['Cat', 'Dog'])}")
print(f"\n✅ All files saved to Google Drive!")
print(f"📂 Google Drive → dogs_vs_cats_results/")
print(f"   - dogs_vs_cats_mobilenetv2.keras")
print(f"   - training_history.png")
print(f"   - confusion_matrix.png")